# Task 2 — Qwen3-VL Visual-CoT + SAM2.1 Submission

`v1 (1).ipynb`의 방법론을 Task 2 reasoning segmentation에 옮긴 버전입니다.

## Pipeline

```text
Original image + query
        ↓
Turn 1 — Qwen3-VL
관련 영역 bbox 최대 3개 탐색
        ↓
crop/zoom
        ↓
Turn 2 — Qwen3-VL
original + crops + query
        ↓
최종 target bbox + positive point
        ↓
SAM2.1
        ↓
segmentation mask
        ↓
1000×1000 bool
        ↓
RLE
        ↓
submission.csv
```

## 제한사항 반영

- 서버에 제공된 **Qwen3-VL-8B-Instruct**만 사용
- 서버에 제공된 **SAM2.1 Hiera Large**만 사용
- 외부 API / 외부 데이터 / 외부 모델 다운로드 없음
- seed = 42 고정
- `torchvision.datasets.ImageFolder()` 사용
- `sample_submission.csv`의 순서/컬럼 유지
- 최종 mask: `[1000, 1000]`, `bool`
- 제공 방식과 동일한 `mask.T.flatten()` RLE 사용
- inference 실패를 임의의 empty mask로 숨기지 않음
- 각 이미지에서 Turn1 bbox, Turn2 최종 bbox/point, SAM score를 debug에 저장


### SAM2 config 주의
현재 서버의 `build_sam2()`는 Hydra config search path를 사용하므로,
checkpoint는 절대경로를 사용하고 config는
`configs/sam2.1/sam2.1_hiera_l.yaml` 이름으로 로드합니다.


In [ ]:
# 실행 순서: 01 → 03 → ensemble 순 실행 필수
# 모델·데이터 경로 하드코딩 — 채점 환경에 맞춘 수정 필요

MODEL_PATH = "/home/jjs2403/2026_bootcamp_02/models/Qwen3-VL-8B-Instruct"

SAM2_MODEL_PATH = (
    "/home/jjs2403/2026_bootcamp_02/models/SAM2.1/"
    "weights/sam2.1_hiera_large.pt"
)

# build_sam2 config 인자는 Hydra config name — yaml 절대경로 전달 시 MissingConfigException
SAM2_CFG_NAME = "configs/sam2.1/sam2.1_hiera_l.yaml"

# 존재 확인 전용 — build_sam2 에 전달 금지
SAM2_CFG_FILE = (
    "/home/jjs2403/2026_bootcamp_02/models/SAM2.1/"
    "weights/sam2.1_hiera_l.yaml"
)

BASE_DIR = (
    "/home/jjs2403/2026_bootcamp_summer/"
    "_ASSIGNMENTS/competitions/competition2"
)

IMAGE_DIR = f"{BASE_DIR}/imgs"
QUERY_CSV = f"{BASE_DIR}/query.csv"

# Jupyter 작업 폴더 기준 상대경로
SAMPLE_SUB_CSV = "./sample_submission2.csv"

OUTPUT_CSV = "./submission_task2_baseline.csv"
DEBUG_CSV = "./debug_task2_baseline.csv"

# flat imgs 를 ImageFolder 로 읽기 위한 로컬 symlink view
IMAGEFOLDER_VIEW = "./_competition2_imagefolder_view"

SEED = 42

N_CROPS = 3
CROP_PADDING = 0.10
RESIZE_SIZE = 840

TURN1_MAX_NEW_TOKENS = 256
TURN2_MAX_NEW_TOKENS = 384
FINAL_MAX_NEW_TOKENS = 256

SHOW_EACH = True
SHOW_CROPS = True

# 전체 200장 제출 시 0 / None 고정
START_INDEX = 0
END_INDEX = None

print("MODEL_PATH :", MODEL_PATH)
print("IMAGE_DIR  :", IMAGE_DIR)
print("QUERY_CSV  :", QUERY_CSV)
print("SAMPLE CSV :", SAMPLE_SUB_CSV)
print("OUTPUT CSV :", OUTPUT_CSV)
print("SAM2 CFG   :", SAM2_CFG_NAME)


In [ ]:
import os
import re
import json
import random
import shutil
from pathlib import Path
from typing import List, Optional, Tuple

import numpy as np
import pandas as pd
import torch

from PIL import Image, ImageDraw
from IPython.display import display
from tqdm.auto import tqdm

from torchvision.datasets import ImageFolder

from transformers import AutoProcessor, Qwen3VLForConditionalGeneration

from sam2.sam2_image_predictor import SAM2ImagePredictor
from sam2.build_sam import build_sam2

def set_seed(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# 대회 안내의 ImageFolder 사용 조건 충족 목적
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def list_images(root: Path):
    return [
        p for p in root.rglob("*")
        if p.is_file() and p.suffix.lower() in IMAGE_EXTS
    ]

image_dir = Path(IMAGE_DIR)
query_csv = Path(QUERY_CSV)
sample_csv = Path(SAMPLE_SUB_CSV)

assert image_dir.exists(), f"IMAGE_DIR 없음: {image_dir}"
assert query_csv.exists(), f"query.csv 없음: {query_csv}"
assert sample_csv.exists(), f"sample_submission.csv 없음: {sample_csv.resolve()}"

source_images = list_images(image_dir)
assert source_images, f"이미지가 없습니다: {image_dir}"

print("원본 이미지 수:", len(source_images))

def build_test_imagefolder(img_dir: Path, source_images: List[Path]):
    try:
        ds = ImageFolder(str(img_dir))
        if len(ds) == len(source_images) and len(ds) > 0:
            print("ImageFolder: 원본 imgs 구조 직접 사용")
            return ds
    except Exception as e:
        print("원본 imgs를 ImageFolder로 직접 읽지 못함:", repr(e))

    view_root = Path(IMAGEFOLDER_VIEW)
    class_dir = view_root / "test_class"
    class_dir.mkdir(parents=True, exist_ok=True)

    for p in class_dir.iterdir():
        if p.is_symlink() or p.is_file():
            p.unlink()
        elif p.is_dir():
            shutil.rmtree(p)

    def natural_key(p):
        return [
            int(x) if x.isdigit() else x.lower()
            for x in re.split(r"(\d+)", p.name)
        ]

    ordered = sorted(source_images, key=natural_key)

    for idx, src in enumerate(ordered):
        dst = class_dir / f"{idx:04d}__{src.name}"
        try:
            dst.symlink_to(src.resolve())
        except OSError:
            shutil.copy2(src, dst)

    ds = ImageFolder(str(view_root))
    assert len(ds) == len(source_images)

    print("ImageFolder local view:", view_root.resolve())
    return ds

test_dataset = build_test_imagefolder(
    image_dir,
    source_images,
)

image_paths = [
    Path(path).resolve()
    for path, _ in test_dataset.samples
]

query_df = pd.read_csv(query_csv)
sample_df = pd.read_csv(sample_csv)

print("ImageFolder images:", len(test_dataset))
print("query rows        :", len(query_df))
print("submission rows   :", len(sample_df))

assert len(test_dataset) == len(query_df), (
    f"image/query 개수 불일치: {len(test_dataset)} != {len(query_df)}"
)

assert len(query_df) == len(sample_df), (
    f"query/submission 개수 불일치: {len(query_df)} != {len(sample_df)}"
)

print("\nquery columns:", list(query_df.columns))
print("sample columns:", list(sample_df.columns))

display(query_df.head())
display(sample_df.head())


In [ ]:
def pick_column(columns, candidates, required=True):
    columns = list(columns)
    lower = {str(c).lower(): c for c in columns}

    for cand in candidates:
        if cand.lower() in lower:
            return lower[cand.lower()]

    for c in columns:
        cl = str(c).lower()
        if any(cand.lower() in cl for cand in candidates):
            return c

    if required:
        raise KeyError(
            f"컬럼을 찾지 못했습니다. "
            f"candidates={candidates}, columns={columns}"
        )

    return None

QUERY_COL = pick_column(
    query_df.columns,
    ["query", "question", "text", "prompt", "sentence"],
)

QUERY_ID_COL = pick_column(
    query_df.columns,
    ["ID", "image_id", "image", "filename", "file_name", "img", "id"],
    required=False,
)

SAMPLE_ID_COL = pick_column(
    sample_df.columns,
    ["ID", "image_id", "image", "filename", "file_name", "img", "id"],
    required=False,
)

SAMPLE_LABEL_COL = pick_column(
    sample_df.columns,
    ["Label", "label", "EncodedPixels", "encoded_pixels", "mask"],
)

print("QUERY_COL       :", QUERY_COL)
print("QUERY_ID_COL    :", QUERY_ID_COL)
print("SAMPLE_ID_COL   :", SAMPLE_ID_COL)
print("SAMPLE_LABEL_COL:", SAMPLE_LABEL_COL)

def lookup_keys(x):
    s = str(x).strip().replace("\\", "/")
    name = Path(s).name
    stem = Path(name).stem

    return {
        s.lower(),
        name.lower(),
        stem.lower(),
    }

image_lookup = {}

for p in image_paths:
    for k in {p.name.lower(), p.stem.lower()}:
        image_lookup.setdefault(k, []).append(p)

def lookup_image(value):
    if pd.isna(value):
        return None

    found = []

    for key in lookup_keys(value):
        found.extend(image_lookup.get(key, []))

    unique = []
    seen = set()

    for p in found:
        s = str(p)
        if s not in seen:
            unique.append(p)
            seen.add(s)

    if len(unique) == 1:
        return unique[0]

    if len(unique) > 1:
        raise RuntimeError(
            f"하나의 ID가 여러 이미지와 매칭됨: {value} -> {unique}"
        )

    return None

def natural_key(p):
    return [
        int(x) if x.isdigit() else x.lower()
        for x in re.split(r"(\d+)", p.name)
    ]

natural_sorted_images = sorted(
    image_paths,
    key=natural_key,
)

# 이미지 매칭 우선순위: query.csv ID → sample_submission ID → natural sort
def resolve_image(row_idx):
    if QUERY_ID_COL is not None:
        p = lookup_image(
            query_df.iloc[row_idx][QUERY_ID_COL]
        )
        if p is not None:
            return p, "query_id"

    if SAMPLE_ID_COL is not None:
        p = lookup_image(
            sample_df.iloc[row_idx][SAMPLE_ID_COL]
        )
        if p is not None:
            return p, "sample_id"

    return natural_sorted_images[row_idx], "natural_sort_fallback"

mapping = []

for i in range(len(query_df)):
    p, method = resolve_image(i)

    mapping.append({
        "row": i,
        "image": p.name,
        "method": method,
        "query": str(query_df.iloc[i][QUERY_COL]),
    })

mapping_df = pd.DataFrame(mapping)

print("\n매칭 방식:")
display(
    mapping_df["method"]
    .value_counts()
    .rename_axis("method")
    .reset_index(name="count")
)

print("\n첫 20개:")
display(mapping_df.head(20))

if (mapping_df["method"] == "natural_sort_fallback").any():
    print(
        "WARNING: ID 기반 매칭이 불가능한 행은 natural sort를 사용합니다."
    )


In [ ]:
def normalize_bbox(
    bbox: List[float],
    img_width: int,
    img_height: int,
) -> List[int]:
    """
    v1과 같은 0~1000 normalized bbox를 원본 픽셀 좌표로 변환.
    안전성을 위해 좌표 정렬/clamp만 추가.
    """
    x1, y1, x2, y2 = map(float, bbox)

    # 프롬프트가 0~1000 정규화를 요구하므로 1000 이하는 정규화 좌표로 간주
    if max(abs(v) for v in [x1, y1, x2, y2]) <= 1000:
        x1 = x1 * img_width / 1000.0
        y1 = y1 * img_height / 1000.0
        x2 = x2 * img_width / 1000.0
        y2 = y2 * img_height / 1000.0

    x1, x2 = sorted([int(round(x1)), int(round(x2))])
    y1, y2 = sorted([int(round(y1)), int(round(y2))])

    x1 = max(0, min(img_width - 1, x1))
    x2 = max(0, min(img_width - 1, x2))
    y1 = max(0, min(img_height - 1, y1))
    y2 = max(0, min(img_height - 1, y2))

    if x2 <= x1:
        x2 = min(img_width - 1, x1 + 1)

    if y2 <= y1:
        y2 = min(img_height - 1, y1 + 1)

    return [x1, y1, x2, y2]

def normalize_point(
    point: List[float],
    img_width: int,
    img_height: int,
) -> List[int]:
    x, y = map(float, point)

    if max(abs(x), abs(y)) <= 1000:
        x = x * img_width / 1000.0
        y = y * img_height / 1000.0

    x = max(0, min(img_width - 1, int(round(x))))
    y = max(0, min(img_height - 1, int(round(y))))

    return [x, y]

def crop_image_with_bbox(
    image: Image.Image,
    bbox: List[int],
    padding: float = CROP_PADDING,
) -> Image.Image:
    """
    v1의 crop 방식:
    bbox 주변 padding + 너무 작으면 최소 영역 확보.
    """
    x1, y1, x2, y2 = bbox

    width = x2 - x1
    height = y2 - y1

    pad_w = int(width * padding)
    pad_h = int(height * padding)

    x1 = max(0, x1 - pad_w)
    y1 = max(0, y1 - pad_h)
    x2 = min(image.width, x2 + pad_w)
    y2 = min(image.height, y2 + pad_h)

    crop_width = x2 - x1
    crop_height = y2 - y1

    if crop_width < 224 or crop_height < 224:
        center_x = (x1 + x2) // 2
        center_y = (y1 + y2) // 2

        half_size = max(
            crop_width,
            crop_height,
            224,
        ) // 2

        x1 = max(0, center_x - half_size)
        y1 = max(0, center_y - half_size)
        x2 = min(image.width, center_x + half_size)
        y2 = min(image.height, center_y + half_size)

    return image.crop((x1, y1, x2, y2))

BOX_PATTERN = (
    r"<\|box_start\|>\s*"
    r"\(([\d.]+),\s*([\d.]+)\),\s*"
    r"\(([\d.]+),\s*([\d.]+)\)"
    r"\s*<\|box_end\|>"
)

JSON_BBOX_PATTERN = (
    r"\[\s*(-?[\d.]+)\s*,\s*"
    r"(-?[\d.]+)\s*,\s*"
    r"(-?[\d.]+)\s*,\s*"
    r"(-?[\d.]+)\s*\]"
)

def extract_bboxes_from_text(
    text: str,
    max_n: int = N_CROPS,
) -> List[List[float]]:
    """
    v1과 동일한 방식:
    Qwen native box token 우선 → 일반 [x1,y1,x2,y2] fallback.
    """
    boxes = [
        [float(v) for v in m]
        for m in re.findall(BOX_PATTERN, text)
    ]

    if not boxes:
        boxes = [
            [float(v) for v in m]
            for m in re.findall(JSON_BBOX_PATTERN, text)
        ]

    return boxes[:max_n]


In [ ]:
@torch.inference_mode()
def run_qwen_inference(
    model,
    processor,
    images,
    prompt,
    max_new_tokens=256,
):
    content = []

    for img in images:
        resized = img.resize(
            (RESIZE_SIZE, RESIZE_SIZE),
            Image.Resampling.BILINEAR,
        )
        content.append({
            "type": "image",
            "image": resized,
        })

    content.append({
        "type": "text",
        "text": prompt,
    })

    message = [{
        "role": "user",
        "content": content,
    }]

    chat_prompt = processor.apply_chat_template(
        message,
        tokenize=False,
        add_generation_prompt=True,
    )

    resized_images = [
        img.resize(
            (RESIZE_SIZE, RESIZE_SIZE),
            Image.Resampling.BILINEAR,
        )
        for img in images
    ]

    inputs = processor(
        text=[chat_prompt],
        images=resized_images,
        return_tensors="pt",
    )

    device = next(model.parameters()).device

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    generated = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        use_cache=True,
    )

    trimmed = generated[
        :,
        inputs["input_ids"].shape[-1]:
    ]

    response = processor.batch_decode(
        trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0]

    return response.strip()


In [ ]:
# 프롬프트 3종 전체 이미지 공통 고정 — 쿼리 원문만 치환
TURN1_PROMPT_TEMPLATE = """You are a visual reasoning assistant.

Reasoning segmentation query:
{query}

Identify up to 3 image regions that are most useful for solving this query.
The regions may contain:
- the target object,
- a reference object mentioned in the query,
- an attribute needed to distinguish the target,
- or spatial/relational evidence needed to identify the target.

Do NOT answer with a segmentation mask yet.
Return up to 3 bounding boxes using coordinates normalized to [0, 1000]
relative to the ORIGINAL IMAGE.

Use this format:

Reasoning Summary: <brief summary>
Bounding Box 1: [x1, y1, x2, y2]
Bounding Box 2: [x1, y1, x2, y2]
Bounding Box 3: [x1, y1, x2, y2]

If fewer than 3 regions are useful, return only the useful boxes.
"""

TURN2_PROMPT_TEMPLATE = """You are solving a reasoning-based image segmentation task.

The FIRST image is the ORIGINAL full image.
The remaining images, if present, are zoomed crops selected from the original image.

Reasoning segmentation query:
{query}

Use the crops only as additional visual evidence.
Identify the exact object or objects in the FIRST / ORIGINAL image that satisfy the query.

IMPORTANT COORDINATE RULE:
All final bbox and point coordinates MUST be normalized to [0, 1000]
relative to the FIRST / ORIGINAL image, NOT relative to any crop.

For each final target:
- bbox_2d = [x1, y1, x2, y2]
- point_2d = one positive point clearly inside that target

Return a brief reasoning summary, then end with ONLY this machine-readable block:

<answer>
{{
  "target_description": "<short target description>",
  "annotations": [
    {{
      "bbox_2d": [x1, y1, x2, y2],
      "point_2d": [x, y]
    }}
  ]
}}
</answer>
"""

FINAL_PROMPT_TEMPLATE = """You are solving a reasoning-based image segmentation task.

Reasoning segmentation query:
{query}

Look only at this original image and identify the exact target object or objects.

All coordinates MUST be normalized to [0, 1000] relative to this image.

For every final target return:
- bbox_2d = [x1, y1, x2, y2]
- point_2d = one positive point clearly inside the target

Return ONLY:

<answer>
{{
  "target_description": "<short target description>",
  "annotations": [
    {{
      "bbox_2d": [x1, y1, x2, y2],
      "point_2d": [x, y]
    }}
  ]
}}
</answer>
"""

def make_turn1_prompt(query):
    return TURN1_PROMPT_TEMPLATE.format(
        query=str(query).strip()
    )

def make_turn2_prompt(query):
    return TURN2_PROMPT_TEMPLATE.format(
        query=str(query).strip()
    )

def make_final_prompt(query):
    return FINAL_PROMPT_TEMPLATE.format(
        query=str(query).strip()
    )


In [ ]:
def _extract_answer_payload(text: str):
    m = re.search(
        r"<answer>\s*(.*?)\s*</answer>",
        text,
        flags=re.S | re.I,
    )

    payload = m.group(1) if m else text
    payload = payload.strip()

    payload = re.sub(
        r"^```(?:json)?\s*",
        "",
        payload,
        flags=re.I,
    )

    payload = re.sub(
        r"\s*```$",
        "",
        payload,
    )

    return payload.strip()

def parse_final_localization(
    text: str,
    image: Image.Image,
):
    """
    Turn2 / Final 출력에서:
    target_description
    bboxes(pixel)
    points(pixel)
    를 추출.
    """
    payload = _extract_answer_payload(text)

    obj = None

    try:
        obj = json.loads(payload)
    except Exception:
        m = re.search(
            r"\{.*\}",
            payload,
            flags=re.S,
        )
        if m:
            try:
                obj = json.loads(m.group(0))
            except Exception:
                pass

    target_description = ""

    annotations = []

    if isinstance(obj, dict):
        target_description = str(
            obj.get("target_description", "")
        )

        if isinstance(obj.get("annotations"), list):
            annotations = obj["annotations"]
        elif "bbox_2d" in obj:
            annotations = [obj]

    bboxes = []
    points = []

    for ann in annotations:
        bbox = ann.get("bbox_2d")
        point = ann.get("point_2d")

        if bbox is None or len(bbox) != 4:
            continue

        pixel_bbox = normalize_bbox(
            bbox,
            image.width,
            image.height,
        )

        if point is not None and len(point) == 2:
            pixel_point = normalize_point(
                point,
                image.width,
                image.height,
            )
        else:
            x1, y1, x2, y2 = pixel_bbox
            pixel_point = [
                (x1 + x2) // 2,
                (y1 + y2) // 2,
            ]

        x1, y1, x2, y2 = pixel_bbox
        px, py = pixel_point

        # positive point 가 bbox 밖일 때 bbox 중심으로 대체
        if not (x1 <= px <= x2 and y1 <= py <= y2):
            pixel_point = [
                (x1 + x2) // 2,
                (y1 + y2) // 2,
            ]

        bboxes.append(pixel_bbox)
        points.append(pixel_point)

    if not bboxes:
        raw_boxes = extract_bboxes_from_text(
            text,
            max_n=8,
        )

        for raw_bbox in raw_boxes:
            pixel_bbox = normalize_bbox(
                raw_bbox,
                image.width,
                image.height,
            )

            x1, y1, x2, y2 = pixel_bbox

            bboxes.append(pixel_bbox)
            points.append([
                (x1 + x2) // 2,
                (y1 + y2) // 2,
            ])

    if not bboxes:
        raise ValueError(
            "최종 Qwen 출력에서 target bbox를 추출하지 못했습니다."
        )

    return target_description, bboxes, points


In [ ]:
def draw_boxes(
    image: Image.Image,
    bboxes,
    points=None,
    box_width=5,
):
    vis = image.convert("RGB").copy()
    draw = ImageDraw.Draw(vis)

    colors = [
        "red",
        "lime",
        "cyan",
        "yellow",
        "magenta",
        "orange",
    ]

    for i, bbox in enumerate(bboxes):
        color = colors[i % len(colors)]

        x1, y1, x2, y2 = bbox

        draw.rectangle(
            [x1, y1, x2, y2],
            outline=color,
            width=box_width,
        )

        draw.text(
            (x1 + 5, max(0, y1 - 18)),
            f"box {i+1}",
            fill=color,
        )

        if points is not None and i < len(points):
            px, py = points[i]
            r = max(5, int(min(image.size) * 0.006))

            draw.ellipse(
                [px-r, py-r, px+r, py+r],
                fill="blue",
                outline="white",
                width=2,
            )

    return vis

def overlay_mask(
    image: Image.Image,
    mask: np.ndarray,
    alpha=0.45,
):
    base = np.asarray(
        image.convert("RGB")
    ).astype(np.float32)

    mask = np.asarray(mask).astype(bool)

    out = base.copy()

    out[mask] = (
        (1.0 - alpha) * out[mask]
        + alpha * np.array(
            [0, 255, 0],
            dtype=np.float32,
        )
    )

    return Image.fromarray(
        np.clip(out, 0, 255).astype(np.uint8)
    )

def display_resized(
    image: Image.Image,
    max_side=800,
):
    img = image.copy()

    w, h = img.size
    scale = min(
        1.0,
        max_side / max(w, h),
    )

    if scale < 1:
        img = img.resize(
            (
                int(w * scale),
                int(h * scale),
            ),
            Image.Resampling.LANCZOS,
        )

    display(img)

def show_crop_strip(
    original_vis,
    crops,
):
    print("Original with Turn1 boxes:")
    display_resized(original_vis)

    for idx, crop in enumerate(crops, start=1):
        print(f"crop {idx}: size={crop.size}")
        display_resized(crop, max_side=500)


In [ ]:
@torch.inference_mode()
def run_sam21(
    predictor,
    image: Image.Image,
    bboxes,
    points,
):
    """
    원본 이미지에 대해 각 bbox + positive point로 SAM2.1 실행.
    각 annotation에서 SAM score 최대 mask 선택.
    여러 target이면 logical OR.
    """
    image_np = np.asarray(
        image.convert("RGB")
    )

    predictor.set_image(image_np)

    selected_masks = []
    selected_scores = []

    for bbox, point in zip(bboxes, points):
        masks, scores, _ = predictor.predict(
            point_coords=np.asarray(
                [point],
                dtype=np.float32,
            ),
            point_labels=np.asarray(
                [1],
                dtype=np.int32,
            ),
            box=np.asarray(
                bbox,
                dtype=np.float32,
            ),
            multimask_output=True,
        )

        best_idx = int(np.argmax(scores))

        selected_masks.append(
            masks[best_idx].astype(bool)
        )

        selected_scores.append(
            float(scores[best_idx])
        )

    if not selected_masks:
        raise ValueError("SAM2.1 mask 생성 실패")

    final_mask = np.logical_or.reduce(
        selected_masks
    ).astype(bool)

    return final_mask, selected_masks, selected_scores

def ensure_1000_bool(mask):
    if torch.is_tensor(mask):
        mask = mask.detach().cpu().numpy()

    mask = np.asarray(mask).astype(bool)

    if mask.shape != (1000, 1000):
        pil_mask = Image.fromarray(
            mask.astype(np.uint8) * 255
        )

        pil_mask = pil_mask.resize(
            (1000, 1000),
            Image.Resampling.NEAREST,
        )

        mask = np.asarray(pil_mask) > 0

    return mask.astype(bool)

# RLE 축 전치 필수 — mask.T.flatten() 기준
def encode_single_mask(mask):
    """
    대회 제공 RLE 방식과 동일.
    """
    if torch.is_tensor(mask):
        mask = mask.detach().cpu().numpy()

    mask = np.asarray(mask).astype(bool)

    assert mask.shape == (1000, 1000)

    pixels = mask.T.flatten()

    pixels = np.concatenate([
        [False],
        pixels,
        [False],
    ])

    runs = np.where(
        pixels[1:] != pixels[:-1]
    )[0] + 1

    runs[1::2] -= runs[::2]

    return " ".join(str(x) for x in runs)


In [ ]:
print("Loading Qwen3-VL...")

processor = AutoProcessor.from_pretrained(
    MODEL_PATH
)

try:
    qwen_model = Qwen3VLForConditionalGeneration.from_pretrained(
        MODEL_PATH,
        torch_dtype=(
            torch.bfloat16
            if torch.cuda.is_available()
            else torch.float32
        ),
        device_map="auto",
        attn_implementation="flash_attention_2",
    )
except Exception as e:
    print(
        "flash_attention_2 사용 실패 -> 기본 attention으로 재시도"
    )
    print("reason:", repr(e))

    qwen_model = Qwen3VLForConditionalGeneration.from_pretrained(
        MODEL_PATH,
        torch_dtype=(
            torch.bfloat16
            if torch.cuda.is_available()
            else torch.float32
        ),
        device_map="auto",
    )

qwen_model.eval()

print("Qwen3-VL loaded.")

print("\nLoading SAM2.1...")

print("checkpoint exists:", Path(SAM2_MODEL_PATH).exists())
print("yaml file exists :", Path(SAM2_CFG_FILE).exists())
print("Hydra config name:", SAM2_CFG_NAME)

if not Path(SAM2_MODEL_PATH).exists():
    raise FileNotFoundError(
        f"SAM2 checkpoint 없음: {SAM2_MODEL_PATH}"
    )

# checkpoint 는 절대경로, config 는 Hydra config name 사용
sam2_model = build_sam2(
    SAM2_CFG_NAME,
    SAM2_MODEL_PATH,
    device="cuda" if torch.cuda.is_available() else "cpu",
)

sam_predictor = SAM2ImagePredictor(
    sam2_model
)

print("SAM2.1 loaded.")


In [ ]:
# 200장 실행 전 1장 점검용 셀
CHECK_INDEX = 0

image_path, match_method = resolve_image(
    CHECK_INDEX
)

query = str(
    query_df.iloc[CHECK_INDEX][QUERY_COL]
)

image = Image.open(
    image_path
).convert("RGB")

print("=" * 100)
print("row   :", CHECK_INDEX)
print("match :", match_method)
print("image :", image_path.name)
print("size  :", image.size)
print("query :", query)

turn1_output = run_qwen_inference(
    qwen_model,
    processor,
    [image],
    make_turn1_prompt(query),
    max_new_tokens=TURN1_MAX_NEW_TOKENS,
)

print("\n[TURN 1 RAW]")
print(turn1_output)

turn1_raw_boxes = extract_bboxes_from_text(
    turn1_output,
    N_CROPS,
)

turn1_boxes = [
    normalize_bbox(
        b,
        image.width,
        image.height,
    )
    for b in turn1_raw_boxes
]

print("\n[TURN 1 BBOX]")
for i, b in enumerate(turn1_boxes, start=1):
    print(f"box {i}: {b}")

crops = [
    crop_image_with_bbox(
        image,
        b,
        padding=CROP_PADDING,
    )
    for b in turn1_boxes
]

turn1_vis = draw_boxes(
    image,
    turn1_boxes,
)

if SHOW_CROPS:
    show_crop_strip(
        turn1_vis,
        crops,
    )

turn2_output = run_qwen_inference(
    qwen_model,
    processor,
    [image] + crops,
    make_turn2_prompt(query),
    max_new_tokens=TURN2_MAX_NEW_TOKENS,
)

print("\n[TURN 2 RAW]")
print(turn2_output)

try:
    (
        target_description,
        final_boxes,
        final_points,
    ) = parse_final_localization(
        turn2_output,
        image,
    )

    used_stage = "turn2"

except Exception as e:
    print("\nTurn2 parsing 실패:", repr(e))
    print("원본-only direct localization으로 재시도")

    final_output = run_qwen_inference(
        qwen_model,
        processor,
        [image],
        make_final_prompt(query),
        max_new_tokens=FINAL_MAX_NEW_TOKENS,
    )

    print("\n[FINAL RAW]")
    print(final_output)

    (
        target_description,
        final_boxes,
        final_points,
    ) = parse_final_localization(
        final_output,
        image,
    )

    used_stage = "direct_fallback"

print("\n[FINAL LOCALIZATION]")
print("stage :", used_stage)
print("target:", target_description)

for i, (b, p) in enumerate(
    zip(final_boxes, final_points),
    start=1,
):
    print(f"target {i}: bbox={b}, point={p}")

final_box_vis = draw_boxes(
    image,
    final_boxes,
    final_points,
)

display_resized(final_box_vis)

(
    final_mask,
    individual_masks,
    sam_scores,
) = run_sam21(
    sam_predictor,
    image,
    final_boxes,
    final_points,
)

print("\n[SAM2.1]")
for i, (score, m) in enumerate(
    zip(sam_scores, individual_masks),
    start=1,
):
    print(
        f"target {i}: score={score:.4f}, "
        f"area={m.mean():.4%}"
    )

print("final mask area:", f"{final_mask.mean():.4%}")

mask_vis = overlay_mask(
    final_box_vis,
    final_mask,
)

display_resized(mask_vis)

submission_mask = ensure_1000_bool(
    final_mask
)

print(
    "\nsubmission mask:",
    submission_mask.shape,
    submission_mask.dtype,
)

print(
    "RLE length:",
    len(encode_single_mask(submission_mask)),
)


In [ ]:
results = []
rle_by_row = {}

start = START_INDEX

end = (
    len(query_df)
    if END_INDEX is None
    else min(END_INDEX, len(query_df))
)

for row_idx in tqdm(
    range(start, end),
    total=end-start,
):

    image_path, match_method = resolve_image(
        row_idx
    )

    query = str(
        query_df.iloc[row_idx][QUERY_COL]
    )

    print("\n\n" + "=" * 110)
    print(f"[{row_idx+1}/{len(query_df)}]")
    print("image :", image_path.name)
    print("match :", match_method)
    print("query :", query)

    turn1_output = ""
    turn2_output = ""
    final_output = ""

    turn1_boxes = []
    final_boxes = []
    final_points = []

    target_description = ""
    sam_scores = []

    final_mask = None
    submission_mask = None
    rle = None

    used_stage = ""
    error = ""

    try:
        image = Image.open(
            image_path
        ).convert("RGB")

        turn1_output = run_qwen_inference(
            qwen_model,
            processor,
            [image],
            make_turn1_prompt(query),
            max_new_tokens=TURN1_MAX_NEW_TOKENS,
        )

        print("\n[1] TURN1 RAW")
        print(turn1_output)

        raw_boxes = extract_bboxes_from_text(
            turn1_output,
            N_CROPS,
        )

        turn1_boxes = [
            normalize_bbox(
                b,
                image.width,
                image.height,
            )
            for b in raw_boxes
        ]

        print("\n[2] TURN1 REGION BBOX")

        for i, b in enumerate(
            turn1_boxes,
            start=1,
        ):
            print(f"region {i}: {b}")

        crops = [
            crop_image_with_bbox(
                image,
                b,
                padding=CROP_PADDING,
            )
            for b in turn1_boxes
        ]

        # 시각화는 첫 이미지 1장만 출력 — 200장 전체 출력 방지
        if SHOW_EACH and row_idx == start:
            turn1_vis = draw_boxes(
                image,
                turn1_boxes,
            )

            print("\n[3] TURN1 BBOX VISUALIZATION")
            display_resized(turn1_vis)

            if SHOW_CROPS:
                for i, crop in enumerate(
                    crops,
                    start=1,
                ):
                    print(f"crop {i}: {crop.size}")
                    display_resized(
                        crop,
                        max_side=450,
                    )

        turn2_output = run_qwen_inference(
            qwen_model,
            processor,
            [image] + crops,
            make_turn2_prompt(query),
            max_new_tokens=TURN2_MAX_NEW_TOKENS,
        )

        print("\n[4] TURN2 RAW")
        print(turn2_output)

        try:
            (
                target_description,
                final_boxes,
                final_points,
            ) = parse_final_localization(
                turn2_output,
                image,
            )

            used_stage = "turn2"

        except Exception as turn2_error:
            print(
                "\nTurn2 localization parse 실패:",
                repr(turn2_error),
            )

            final_output = run_qwen_inference(
                qwen_model,
                processor,
                [image],
                make_final_prompt(query),
                max_new_tokens=FINAL_MAX_NEW_TOKENS,
            )

            print("\n[5] DIRECT FALLBACK RAW")
            print(final_output)

            (
                target_description,
                final_boxes,
                final_points,
            ) = parse_final_localization(
                final_output,
                image,
            )

            used_stage = "direct_fallback"

        print("\n[6] FINAL TARGET LOCALIZATION")
        print("stage :", used_stage)
        print("target:", target_description)

        for i, (b, p) in enumerate(
            zip(final_boxes, final_points),
            start=1,
        ):
            print(
                f"target {i}: bbox={b}, point={p}"
            )

        if SHOW_EACH and row_idx == start:
            final_box_vis = draw_boxes(
                image,
                final_boxes,
                final_points,
            )

            display_resized(final_box_vis)

        (
            final_mask,
            individual_masks,
            sam_scores,
        ) = run_sam21(
            sam_predictor,
            image,
            final_boxes,
            final_points,
        )

        print("\n[7] SAM2.1")

        for i, (score, mask_i) in enumerate(
            zip(sam_scores, individual_masks),
            start=1,
        ):
            print(
                f"target {i}: "
                f"score={score:.4f}, "
                f"area={mask_i.mean():.4%}"
            )

        print(
            "final area:",
            f"{final_mask.mean():.4%}",
        )

        if SHOW_EACH and row_idx == start:
            final_box_vis = draw_boxes(
                image,
                final_boxes,
                final_points,
            )

            mask_vis = overlay_mask(
                final_box_vis,
                final_mask,
            )

            print("\n[8] FINAL MASK")
            display_resized(mask_vis)

        submission_mask = ensure_1000_bool(
            final_mask
        )

        rle = encode_single_mask(
            submission_mask
        )

        rle_by_row[row_idx] = rle

    except Exception as e:
        error = repr(e)

        print("\n!!! ERROR !!!")
        print(error)

        # 실패 row 는 None 유지 — 임의 empty mask 대체 금지
        rle_by_row[row_idx] = None

    results.append({
        "row_index": row_idx,
        "image": image_path.name,
        "match_method": match_method,
        "query": query,
        "turn1_output": turn1_output,
        "turn1_bboxes": json.dumps(
            turn1_boxes,
            ensure_ascii=False,
        ),
        "turn2_output": turn2_output,
        "direct_fallback_output": final_output,
        "used_stage": used_stage,
        "target_description": target_description,
        "final_bboxes": json.dumps(
            final_boxes,
            ensure_ascii=False,
        ),
        "final_points": json.dumps(
            final_points,
            ensure_ascii=False,
        ),
        "sam_scores": json.dumps(
            sam_scores,
        ),
        "mask_area_ratio": (
            float(submission_mask.mean())
            if submission_mask is not None
            else np.nan
        ),
        "rle_length": (
            len(rle)
            if isinstance(rle, str)
            else np.nan
        ),
        "error": error,
    })

    if (
        ((row_idx - start + 1) % 10 == 0)
        or row_idx == end - 1
    ):
        pd.DataFrame(
            results
        ).to_csv(
            DEBUG_CSV,
            index=False,
        )

        print(
            "\ndebug checkpoint:",
            Path(DEBUG_CSV).resolve(),
        )

print("\nInference finished.")
print("processed:", len(results))


In [ ]:
debug_df = pd.DataFrame(results)

failed_df = debug_df[
    debug_df["error"] != ""
].copy()

print("processed:", len(debug_df))
print("failures :", len(failed_df))

if len(debug_df) > 0:
    print(
        "mean mask area:",
        debug_df["mask_area_ratio"].mean(),
    )

    print("\nused stage:")
    display(
        debug_df["used_stage"]
        .value_counts(dropna=False)
        .rename_axis("stage")
        .reset_index(name="count")
    )

if len(failed_df) > 0:
    print("\n실패 sample:")
    display(
        failed_df[
            [
                "row_index",
                "image",
                "query",
                "turn1_output",
                "turn2_output",
                "direct_fallback_output",
                "error",
            ]
        ]
    )
else:
    print("\n모든 sample inference 성공")


In [ ]:
# 실패가 하나라도 있으면 저장 중단
if start != 0 or end != len(query_df):
    raise RuntimeError(
        "최종 제출은 START_INDEX=0, END_INDEX=None으로 "
        "전체 test를 실행한 뒤 생성하세요."
    )

failed_rows = [
    i for i in range(len(sample_df))
    if rle_by_row.get(i) is None
]

if failed_rows:
    raise RuntimeError(
        f"{len(failed_rows)}개 inference 실패. "
        f"빈 mask로 자동 대체하지 않습니다. "
        f"실패 row={failed_rows}"
    )

submission = sample_df.copy()

submission[SAMPLE_LABEL_COL] = [
    rle_by_row[i]
    for i in range(len(submission))
]

assert len(submission) == 200, (
    f"제출 row가 200이 아닙니다: {len(submission)}"
)

assert not submission[
    SAMPLE_LABEL_COL
].isna().any()

assert submission[
    SAMPLE_LABEL_COL
].map(
    lambda x: isinstance(x, str)
).all()

submission.to_csv(
    OUTPUT_CSV,
    index=False,
)

print("saved:", Path(OUTPUT_CSV).resolve())
print("rows :", len(submission))
print("label column:", SAMPLE_LABEL_COL)

if SAMPLE_ID_COL is not None:
    print(
        "ID duplicated:",
        submission[SAMPLE_ID_COL].duplicated().sum(),
    )

display(submission.head(20))


In [ ]:
final_df = pd.read_csv(
    OUTPUT_CSV
)

print("path:", Path(OUTPUT_CSV).resolve())
print("rows:", len(final_df))
print("columns:", list(final_df.columns))

print(
    "missing label:",
    final_df[SAMPLE_LABEL_COL].isna().sum(),
)

print(
    "empty RLE strings:",
    (
        final_df[SAMPLE_LABEL_COL]
        .fillna("")
        .astype(str)
        .str.len()
        == 0
    ).sum(),
)

print("\nDebug CSV:")
print(Path(DEBUG_CSV).resolve())

display(final_df.head())
